In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.52.4

Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 67.9 MB/s eta 0:00:00


In [ ]:
!pip uninstall -y torch
!pip install torch==2.6.0

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.7 MB/s eta 0:00:00

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
import torch
from datasets import Dataset

# 데이터프레임으로 파일 읽기
df = pd.read_csv('/content/drive/MyDrive/디스부 최종 프로젝트(온라인 그루밍 범죄 탐지)/cleaned_labeling_[SEP]Delete.csv')

In [ ]:
invalid_rows = df[~df['text'].str.startswith(('0:', '1:'))]

In [ ]:
print(f"잘못된 시작 형식의 행 개수: {len(invalid_rows)}")

잘못된 시작 형식의 행 개수: 12


In [ ]:
df_cleaned = df[df['text'].str.startswith(('0:', '1:'))].copy()

In [ ]:
import re

def fix_dialogue(text):
    match = re.search(r'(0:|1:)', text)
    if match:
        return text[match.start():].strip()
    else:
        return None  # 완전 무효일 경우

df['text_fixed'] = df['text'].apply(fix_dialogue)
df = df[df['text_fixed'].notnull()].copy()  # 유효한 행만 남기기
df['text'] = df['text_fixed']
df.drop(columns=['text_fixed'], inplace=True)


In [ ]:
still_invalid = df[~df['text'].str.startswith(('0:', '1:'))]
print(f"📌 전처리 후 남은 잘못된 형식: {len(still_invalid)}")

📌 전처리 후 남은 잘못된 형식: 0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29520 entries, 0 to 29520
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    29520 non-null  object
 1   label   29520 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 691.9+ KB


In [ ]:
df = df.reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29520 entries, 0 to 29519
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    29520 non-null  object
 1   label   29520 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 461.4+ KB


In [ ]:
# Tokenizer 준비

from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

special_tokens_dict = {
    'additional_special_tokens': ['<s_speaker0>', '<s_speaker1>']
}
num_added_tokens = tokenizer.add_special_tokens(special_tokens_dict)
print(f"✅ {num_added_tokens} special tokens added to tokenizer.")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

✅ 2 special tokens added to tokenizer.


In [ ]:
# 원본 대화 단위 split

from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

print(f"✅ Train 대화 수: {len(train_df)}, Val 대화 수: {len(val_df)}")

In [ ]:
# parse_dialogue_with_special_tokens 적용

def parse_dialogue_with_special_tokens(dialogue_text):
    import re
    splits = re.split(r'(?=(?:0:|1:))', dialogue_text)
    turns = [s.strip() for s in splits if s.strip() != '']

    processed_text = ''
    for turn in turns:
        if turn.startswith('0:'):
            turn_text = turn[2:].strip()
            processed_text += ' <s_speaker0> ' + turn_text
        elif turn.startswith('1:'):
            turn_text = turn[2:].strip()
            processed_text += ' <s_speaker1> ' + turn_text

    return processed_text.strip()

# 적용
train_df['parsed_text'] = train_df['text'].apply(parse_dialogue_with_special_tokens)
val_df['parsed_text'] = val_df['text'].apply(parse_dialogue_with_special_tokens)

In [ ]:
# Token sliding 적용

def create_token_sliding_inputs_for_training(texts, labels, tokenizer, window_size=512, stride=384):
    input_texts = []
    target_labels = []

    for text, label in zip(texts, labels):
        inputs = tokenizer(text, truncation=False, return_tensors='pt')
        input_ids = inputs['input_ids'][0]

        seq_len = len(input_ids)

        for i in range(0, seq_len, stride):
            input_ids_chunk = input_ids[i:i + window_size]

            if len(input_ids_chunk) < 10:
                continue

            chunk_text = tokenizer.decode(input_ids_chunk, skip_special_tokens=False)

            input_texts.append(chunk_text)
            target_labels.append(label)

            if i + window_size >= seq_len:
                break

    return input_texts, target_labels

# Sliding 적용
train_input_texts, train_target_labels = create_token_sliding_inputs_for_training(
    train_df['parsed_text'],
    train_df['label'],
    tokenizer,
    window_size=512,
    stride=384
)

val_input_texts, val_target_labels = create_token_sliding_inputs_for_training(
    val_df['parsed_text'],
    val_df['label'],
    tokenizer,
    window_size=512,
    stride=384
)

print(f"✅ Train window 수: {len(train_input_texts)}, Val window 수: {len(val_input_texts)}")


In [ ]:
# Huggingface Dataset 구성

from datasets import Dataset

train_dataset = Dataset.from_dict({
    'text': train_input_texts,
    'label': train_target_labels,
    'contrastive_labels': train_target_labels
})

val_dataset = Dataset.from_dict({
    'text': val_input_texts,
    'label': val_target_labels,
    'contrastive_labels': val_target_labels
})

# Tokenizer 적용
def tokenize_function(examples):
    return tokenizer(examples['text'], max_length=512, truncation=True, padding='max_length')

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)


In [ ]:
# 모델 선언

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.resize_token_embeddings(len(tokenizer))
print(f"✅ Model embedding size resized to {len(tokenizer)}.")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


✅ Model embedding size resized to 30524.


In [ ]:
# 평가 함수
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average='binary')  # 다중 분류면 average='weighted' 또는 'macro'
    recall = recall_score(labels, preds, average='binary')
    f1 = f1_score(labels, preds, average='binary')

    # AUC-ROC 계산 (이진 분류일 경우)
    try:
        probs = logits[:, 1]  # 클래스 1에 대한 확률 (softmax 생략 가능, ranking에 영향을 안 줌)
        auc = roc_auc_score(labels, probs)
    except:
        auc = float('nan')  # 예외처리: AUC 계산 불가 시

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    }

In [ ]:
# # Class Weights 설정 → Data Re-weighting 적용

# import torch

# # 예시 → Negative:Positive ≈ 20000:3000 → 6.67배 imbalance → weight 5~6 정도로 tuning
# class_weights = torch.tensor([1.0, 5.0]).to('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# # Data Re-weighting 적용 train

# from transformers import TrainingArguments, Trainer

# training_args = TrainingArguments(
#     output_dir='./results_bert_reweight_only',
#     evaluation_strategy="epoch",
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     logging_dir='./logs_bert_reweight_only',
#     logging_steps=10,
#     save_safetensors=False,
# )

# # Trainer 정의
# class TrainerWithClassWeights(Trainer):
#     def compute_loss(self, model, inputs, return_outputs=False):
#         labels = inputs.get("labels")
#         outputs = model(
#             input_ids=inputs["input_ids"],
#             attention_mask=inputs["attention_mask"],
#             labels=labels
#         )
#         logits = outputs.logits

#         loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
#         loss = loss_fct(logits, labels)

#         return (loss, outputs) if return_outputs else loss

# # Trainer 선언
# trainer = TrainerWithClassWeights(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     tokenizer=tokenizer
# )

# # 학습 시작
# trainer.train()


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./results_bert',
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir='./logs_bert',
    logging_steps=10,
    save_safetensors=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics  # 평가를 위해 추가
)

trainer.train()

<ipython-input-21-447078b0c3a6>:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shjeong020208 (shjeong020208-dong-eui-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.000900,0.011337


TrainOutput(global_step=2298, training_loss=0.028819056247813696, metrics={'train_runtime': 3846.9506, 'train_samples_per_second': 9.557, 'train_steps_per_second': 0.597, 'total_flos': 9673014839255040.0, 'train_loss': 0.028819056247813696, 'epoch': 1.0})

In [ ]:
import torch
import torch.nn as nn
from transformers import BartModel, BartTokenizerFast
import os

# Create the directory if it doesn't exist
save_directory = '/content/drive/MyDrive/BERT_Fine_Tuning_GroomingData_SlidingWindow_Model'
if not os.path.exists(save_directory):
    os.makedirs(save_directory)
    print(f"Directory '{save_directory}' created.")
else:
    print(f"Directory '{save_directory}' already exists.")


# 모델 저장
model.save_pretrained('/content/drive/MyDrive/BERT_Fine_Tuning_GroomingData_SlidingWindow_Model')
tokenizer.save_pretrained('/content/drive/MyDrive/BERT_Fine_Tuning_GroomingData_SlidingWindow_Model')

print("\nModel and tokenizer saved successfully.")

Directory '/content/drive/MyDrive/BERT_Fine_Tuning_GroomingData_SlidingWindow_Model' created.

Model and tokenizer saved successfully.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import numpy as np
import torch

outputs = trainer.predict(val_dataset)

probs = torch.softmax(torch.tensor(outputs.predictions), dim=-1).numpy()
true_labels = np.array(outputs.label_ids)

thresholds = np.arange(0.05, 0.95, 0.05)
best_f1 = 0
best_threshold = 0
results = []

for thresh in thresholds:
    pred_labels = (probs[:, 1] >= thresh).astype(int)
    acc = accuracy_score(true_labels, pred_labels)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary')
    auc_roc = roc_auc_score(true_labels, probs[:, 1])
    results.append((thresh, acc, precision, recall, f1, auc_roc))

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thresh

In [ ]:
# 평가
results = trainer.evaluate()
print(results)


=== Evaluation Metrics ===
Accuracy : 0.9979
Precision: 0.9893
Recall   : 0.9950
F1 Score : 0.9921
AUC-ROC  : 0.9999


In [ ]:
from transformers import BertForSequenceClassification, BertTokenizerFast
import torch
import time
import matplotlib.pyplot as plt

# 1️⃣ 모델 로드
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertForSequenceClassification.from_pretrained("/content/bert_grooming_model")  # 저장된 경로에 맞게 수정
tokenizer = BertTokenizerFast.from_pretrained("/content/bert_grooming_model")

model = model.to(device)
model.eval()

# 2️⃣ 실시간 대화 흐름 관리
dialogue_history = []
window_size = 10
# best_threshold = 0.7  # tuning 결과 적용

# Progressive Risk Score Tracking
risk_scores = []

# 3️⃣ Inactivity timer 설정
last_update_time = time.time()
reset_timeout = 180  # 3분 inactivity 시 history reset

# 4️⃣ 실시간 테스트 loop 시작
print("=== Grooming 위험도 실시간 탐지 시작 (BERT 기반) ===")
print("※ 상대방 발화는 '1:' 으로 시작, 내 발화는 '0:' 으로 시작해서 입력해주세요.")
print("※ 'END' 입력 시 종료\n")

while True:
    new_utterance = input("새로운 채팅 입력 (형식: '0: hello' 또는 '1: hi'): ")
    if new_utterance == "END":
        print("💬 실시간 테스트 종료")
        break

    # Inactivity check
    current_time = time.time()
    if current_time - last_update_time > reset_timeout:
        print("💡 대화 inactivity → dialogue_history reset됨.")
        dialogue_history = []
        risk_scores = []  # Progressive tracking도 초기화

    last_update_time = current_time

    # 입력 validation
    if not (new_utterance.startswith("0:") or new_utterance.startswith("1:")):
        print("⚠️ 입력은 반드시 '0:' 또는 '1:' 으로 시작해야 합니다.\n")
        continue

    # dialogue_history 업데이트
    dialogue_history.append(new_utterance.strip())
    if len(dialogue_history) > window_size:
        dialogue_history = dialogue_history[-window_size:]

    # Model input 구성 (parse_dialogue_with_special_tokens 방식 적용)
    model_input_text = ''
    for turn in dialogue_history:
        if turn.startswith("0:"):
            turn_text = turn[2:].strip()
            model_input_text += ' <s_speaker0> ' + turn_text
        elif turn.startswith("1:"):
            turn_text = turn[2:].strip()
            model_input_text += ' <s_speaker1> ' + turn_text
    model_input_text = model_input_text.strip()

    # Tokenization
    inputs = tokenizer(model_input_text, return_tensors='pt', max_length=512, truncation=True, padding='max_length')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Inference
    with torch.no_grad():
        outputs = model(inputs['input_ids'], inputs['attention_mask'])
        logits = outputs['logits']
        probs = torch.softmax(logits, dim=-1)

    grooming_prob = probs[:, 1].item()
    risk_scores.append(grooming_prob)  # Progressive tracking

    # 결과 출력
    print("\n[현재 대화 흐름]")
    for turn in dialogue_history:
        print(turn)

    print(f"\n⚠️ Grooming 위험도 (대화 흐름 기준): {grooming_prob:.3f}")

    if grooming_prob >= best_threshold:
        print("🚨 경고: Grooming 위험 가능성 있음!\n")
    else:
        print("✅ 안전합니다.\n")

# 5️⃣ Progressive Risk Score Tracking 결과 시각화
if len(risk_scores) > 0:
    plt.figure(figsize=(10, 4))
    plt.plot(range(1, len(risk_scores)+1), risk_scores, marker='o')
    plt.axhline(best_threshold, color='r', linestyle='--', label=f"Threshold = {best_threshold}")
    plt.xlabel("대화 단계 (턴)")
    plt.ylabel("Grooming 위험도")
    plt.title("Progressive Risk Score Tracking (BERT)")
    plt.legend()
    plt.grid()
    plt.show()
